# Sentiment Arcs Analysis with Multiple APIs (GLM-4 Enhanced)

**Audience**: Researchers and analysts working with literary texts
**Goal**: Comprehensive sentiment analysis with multiple APIs, configurable smoothing, and detailed peak/valley extraction

### Features:
1. **File Upload**: Clean text file processing
2. **Triple Analysis**: VADER, DistilBERT, and OpenAI API
3. **Configurable Smoothing**: MA, EMA, Savitzky-Golay with adjustable parameters
4. **Peak/Valley Detection**: Automated identification with context extraction
5. **Report Generation**: Downloadable summaries with CRUX_WIDTH context

> **CRUX_WIDTH**: Number of sentences before/after each peak/valley to include in reports

### Analysis Pipeline:
1. Upload and segment text → sentence-level analysis
2. Apply three sentiment methods → convergence analysis
3. Configurable smoothing → parameter optimization
4. Peak/valley detection → context extraction
5. Report generation → downloadable insights

**Note**: This notebook handles OpenAI API integration with proper error handling and fallbacks.

## 1) Setup & Environment

**Enhanced installation** with additional packages for file handling and API integration.

In [ ]:
%%bash
# Enhanced installation for GLM-4 version
echo "Installing enhanced dependencies..."

# Core packages
pip install -q "torch>=2.0" "numpy>=1.24" "pandas>=2.0" --upgrade

# ML and NLP packages
pip install -q "transformers>=4.44,<5" "datasets>=3.0,<3.1" "accelerate>=1.0" --upgrade

# Text processing and sentiment
pip install -q "vaderSentiment>=3.3.2" "textblob>=0.18" "pysbd>=0.3" --upgrade

# Visualization and analysis
pip install -q "matplotlib>=3.9" "plotly>=5.24" "scipy>=1.13" "scikit-learn>=1.5" --upgrade

# File handling and API
pip install -q "openai>=1.0" "ipywidgets>=8.0" "tqdm>=4.66" --upgrade

# Report generation
pip install -q "jupyter>=1.0" "nbformat>=5.0" --upgrade

echo "Downloading NLTK data..."
python - <<'PY'
import nltk
try:
    nltk.download("punkt", quiet=True)
    nltk.download("vader_lexicon", quiet=True)
    print("NLTK data downloaded successfully")
except Exception as e:
    print(f"Warning: NLTK download failed: {e}")
PY

echo "Enhanced installation complete!"

In [ ]:
# Enhanced environment check with API-specific info
import sys, platform, importlib

print("Python:", sys.version)
print("Platform:", platform.platform())

# Check all required libraries
modules_to_check = [
    "torch", "transformers", "openai", "pandas", "numpy", 
    "matplotlib", "scipy", "sklearn", "vaderSentiment", 
    "pysbd", "plotly", "nltk", "ipywidgets"
]

print("\nLibrary Versions:")
print("=" * 20)
for mod in modules_to_check:
    try:
        m = importlib.import_module(mod)
        version = getattr(m, "__version__", "n/a")
        print(f"{mod}: {version}")
    except Exception as e:
        print(f"{mod}: not installed ({e})")

# Check OpenAI configuration
try:
    import openai
    print(f"\nOpenAI version: {openai.__version__}")
    # Note: API key checking will be done later
except Exception as e:
    print(f"OpenAI setup issue: {e}")

print(f"\nEnhanced environment check complete.")

## 2) Enhanced Configuration

**Extended configuration** with OpenAI API settings and CRUX_WIDTH parameter.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np, random, re
import warnings
warnings.filterwarnings("ignore")

@dataclass
class EnhancedConfig:
    # Basic settings
    project_name: str = "SentimentArcs_Enhanced"
    seed: int = 42
    
    # Text processing
    max_sentences: int = 5000  # Limit for API costs
    min_sentence_length: int = 10  # Minimum characters per sentence
    
    # Sentiment analysis models
    vader_enabled: bool = True
    bert_model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"
    openai_enabled: bool = True
    openai_model: str = "gpt-3.5-turbo"  # or "gpt-4" for higher quality
    
    # Smoothing parameters (adjustable)
    smoothing_methods: list = None  # Will be set to ["ma", "ema", "savgol"]
    window_sizes: list = None  # Will be set to [0.05, 0.1, 0.15, 0.2]
    ema_alphas: list = None  # Will be set to [0.1, 0.2, 0.3]
    savgol_polys: list = None  # Will be set to [2, 3, 4]
    
    # Peak/valley detection
    peak_min_frac: float = 0.05
    valley_min_frac: float = 0.05
    peak_prominence: float = 0.3  # Minimum prominence for peaks
    
    # Context extraction
    CRUX_WIDTH: int = 5  # Number of sentences before/after peaks/valleys
    
    # Output settings
    output_dir: str = "./outputs"
    create_reports: bool = True
    interactive_plots: bool = True
    
    def __post_init__(self):
        # Set default lists if None
        if self.smoothing_methods is None:
            self.smoothing_methods = ["ma", "ema", "savgol"]
        if self.window_sizes is None:
            self.window_sizes = [0.05, 0.1, 0.15, 0.2]
        if self.ema_alphas is None:
            self.ema_alphas = [0.1, 0.2, 0.3]
        if self.savgol_polys is None:
            self.savgol_polys = [2, 3, 4]

# Initialize configuration
CFG = EnhancedConfig()

# Seeding
np.random.seed(CFG.seed)
random.seed(CFG.seed)
try:
    import torch
    torch.manual_seed(CFG.seed)
except Exception:
    pass

# Create output directory
out_dir = Path(CFG.output_dir)
out_dir.mkdir(exist_ok=True, parents=True)

print(f"Configuration loaded:")
print(f"- CRUX_WIDTH: {CFG.CRUX_WIDTH} sentences")
print(f"- Max sentences: {CFG.max_sentences}")
print(f"- Smoothing methods: {CFG.smoothing_methods}")
print(f"- Output directory: {out_dir}")

## 3) File Upload and Text Processing

**Interactive file upload** with text cleaning and sentence segmentation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# File upload widget
upload_widget = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Upload Text File',
    style={'description_width': 'initial'}
)

# Text area for manual input
text_widget = widgets.Textarea(
    value='',
    placeholder='Or paste your text here...',
    layout=widgets.Layout(width='100%', height='200px')
)

# Process button
process_button = widgets.Button(
    description='Process Text',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Status output
status_output = widgets.Output()

def clean_text(txt: str) -> str:
    """Enhanced text cleaning"""
    # Remove excessive whitespace
    txt = re.sub(r'\s+', ' ', txt)
    # Remove common artifacts
    txt = re.sub(r'[\t\f\v]', ' ', txt)
    # Clean up quotes
    txt = re.sub(r'[""''`]', '"', txt)
    return txt.strip()

def segment_sentences(txt: str):
    """Enhanced sentence segmentation"""
    try:
        import pysbd
        seg = pysbd.Segmenter(language="en", clean=True)
        sents = [s.strip() for s in seg.segment(txt)]
    except Exception:
        # Fallback: split on punctuation with lookahead
        sents = re.split(r'(?<=[.!?])\s+', txt)
    
    # Filter out very short sentences and clean
    filtered_sents = []
    for s in sents:
        s = s.strip()
        if len(s) >= CFG.min_sentence_length and not s.isdigit():
            filtered_sents.append(s)
    
    return filtered_sents

def process_text(change=None):
    """Main text processing function"""
    with status_output:
        clear_output()
        
        text_content = None
        
        # Try uploaded file first
        if upload_widget.value:
            uploaded_file = list(upload_widget.value.values())[0]
            text_content = uploaded_file['content'].decode('utf-8')
            print(f"✓ Loaded file: {uploaded_file['name']}")
        
        # Fallback to manual input
        elif text_widget.value.strip():
            text_content = text_widget.value.strip()
            print("✓ Using manual text input")
        
        else:
            print("⚠️ Please upload a file or enter text")
            return
        
        # Process the text
        print("🔄 Processing text...")
        
        # Clean and segment
        cleaned_text = clean_text(text_content)
        sentences = segment_sentences(cleaned_text)
        
        # Limit sentences for API cost control
        if len(sentences) > CFG.max_sentences:
            sentences = sentences[:CFG.max_sentences]
            print(f"⚠️ Limited to {CFG.max_sentences} sentences for API efficiency")
        
        # Create DataFrame
        global df
        df = pd.DataFrame({
            'i': range(len(sentences)),
            'sentence': sentences,
            'sentence_length': [len(s) for s in sentences]
        })
        
        print(f"✓ Processed {len(sentences)} sentences")
        print(f"✓ Average sentence length: {df['sentence_length'].mean():.1f} characters")
        print(f"✓ Total characters: {len(cleaned_text):,}")
        
        # Display sample
        print("\n📝 Sample sentences:")
        for i in range(min(3, len(df))):
            print(f"  {i+1}. {df.iloc[i]['sentence'][:100]}...")

# Link the button
process_button.on_click(process_text)

# Display the interface
print("📁 Upload a text file or paste text below:")
display(upload_widget)
print("\n📝 Or paste your text here:")
display(text_widget)
print("\n")
display(process_button)
print("\n")
display(status_output)

## 4) Triple Sentiment Analysis

**Enhanced sentiment analysis** with VADER, DistilBERT, and OpenAI API integration.

### 4.1 VADER Sentiment Analysis

In [ ]:
if 'df' not in globals():
    print("⚠️ Please process text first using the interface above")
else:
    print("🔍 Running VADER sentiment analysis...")
    
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    
    def vader_score(sent: str) -> float:
        """Get VADER compound score"""
        return float(sia.polarity_scores(sent)["compound"])
    
    # Apply VADER to all sentences
    df["vader"] = [vader_score(s) for s in df["sentence"]]
    df["vader_pos"] = [sia.polarity_scores(s)["pos"] for s in df["sentence"]]
    df["vader_neg"] = [sia.polarity_scores(s)["neg"] for s in df["sentence"]]
    df["vader_neu"] = [sia.polarity_scores(s)["neu"] for s in df["sentence"]]
    
    print(f"✓ VADER analysis complete")
    print(f"  Average compound: {df['vader'].mean():.3f}")
    print(f"  Sentiment range: [{df['vader'].min():.3f}, {df['vader'].max():.3f}]")
    
    # Show VADER distribution
    positive = (df['vader'] > 0.05).sum()
    negative = (df['vader'] < -0.05).sum()
    neutral = len(df) - positive - negative
    
    print(f"  Positive: {positive} ({positive/len(df)*100:.1f}%)")
    print(f"  Negative: {negative} ({negative/len(df)*100:.1f}%)")
    print(f"  Neutral: {neutral} ({neutral/len(df)*100:.1f}%)")

### 4.2 DistilBERT Sentiment Analysis

In [ ]:
if 'df' not in globals():
    print("⚠️ Please process text first")
else:
    print("🤖 Running DistilBERT sentiment analysis...")
    
    from transformers import pipeline
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Determine device
    device = -1  # CPU fallback
    try:
        import torch
        if torch.cuda.is_available():
            device = 0
            print("  Using GPU acceleration")
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = "mps"
            print("  Using Apple Silicon GPU")
        else:
            print("  Using CPU")
    except Exception:
        print("  Using CPU (fallback)")
    
    # Initialize pipeline
    clf = pipeline(
        task="sentiment-analysis",
        model=CFG.bert_model_name,
        tokenizer=CFG.bert_model_name,
        truncation=True,
        max_length=256,
        device=device
    )
    
    # Batch processing
    batch_size = 32
    sentences_list = df["sentence"].tolist()
    
    try:
        preds = clf(sentences_list, batch_size=batch_size, padding=True, truncation=True)
    except Exception as e:
        print(f"  Batch processing failed: {e}")
        print("  Falling back to individual processing...")
        preds = clf(sentences_list)
    
    # Convert to signed scores
    label_to_sign = {"NEGATIVE": -1, "POSITIVE": +1}
    
    def signed_score(prediction):
        label = prediction.get("label", "").upper()
        score = float(prediction.get("score", 1.0))
        return label_to_sign.get(label, 0) * score
    
    df["bert_score"] = [signed_score(p) for p in preds]
    df["bert_label"] = [p.get("label", "UNKNOWN") for p in preds]
    df["bert_confidence"] = [float(p.get("score", 0.0)) for p in preds]
    
    print(f"✓ DistilBERT analysis complete")
    print(f"  Average score: {df['bert_score'].mean():.3f}")
    print(f"  Sentiment range: [{df['bert_score'].min():.3f}, {df['bert_score'].max():.3f}]")
    print(f"  Average confidence: {df['bert_confidence'].mean():.3f}")
    
    # Label distribution
    label_counts = df['bert_label'].value_counts()
    for label, count in label_counts.items():
        print(f"  {label}: {count} ({count/len(df)*100:.1f}%)")

### 4.3 OpenAI API Sentiment Analysis

In [ ]:
if 'df' not in globals():
    print("⚠️ Please process text first")
else:
    # Check for OpenAI API key
    try:
        import openai
        
        # Try to get API key from environment
        api_key = None
        try:
            api_key = openai.api_key
        except:
            pass
        
        if not api_key:
            import os
            api_key = os.getenv('OPENAI_API_KEY')
        
        if not api_key:
            print("⚠️ OpenAI API key not found")
            print("Please set OPENAI_API_KEY environment variable")
            print("Or use: openai.api_key = 'your-key-here'")
            
            # Provide option to enter key manually
            print("\n💡 Or enter your API key here:")
            api_key_input = input("OpenAI API Key (press Enter to skip): ").strip()
            if api_key_input:
                api_key = api_key_input
                openai.api_key = api_key
        
        if api_key:
            print("🔑 OpenAI API key found")
            
            # Set up OpenAI client
            client = openai.OpenAI(api_key=api_key)
            
            print("🧠 Running OpenAI sentiment analysis...")
            
            def get_openai_sentiment(sentence, model=CFG.openai_model):
                """Get sentiment from OpenAI API"""
                try:
                    response = client.chat.completions.create(
                        model=model,
                        messages=[
                            {"role": "system", "content": "Rate the sentiment of this sentence on a scale from -1 (very negative) to 1 (very positive). Respond with only the numerical score."},
                            {"role": "user", "content": sentence}
                        ],
                        max_tokens=10,
                        temperature=0
                    )
                    
                    result = response.choices[0].message.content.strip()
                    # Extract numerical value
                    import re
                    match = re.search(r'-?\d*\.?\d+', result)
                    if match:
                        score = float(match.group())
                        # Clamp to [-1, 1]
                        return max(-1.0, min(1.0, score))
                    else:
                        return 0.0
                        
                except Exception as e:
                    print(f"  API call failed: {e}")
                    return 0.0
            
            # Process sentences in smaller batches for rate limiting
            batch_size = 10
            openai_scores = []
            openai_raw = []
            
            for i in range(0, len(df), batch_size):
                batch = df.iloc[i:i+batch_size]
                print(f"  Processing batch {i//batch_size + 1}/{(len(df)-1)//batch_size + 1}...")
                
                batch_scores = []
                batch_raw = []
                
                for _, row in batch.iterrows():
                    sentence = row['sentence']
                    score = get_openai_sentiment(sentence)
                    batch_scores.append(score)
                    batch_raw.append(f"{sentence[:50]}... -> {score:.3f}")
                
                openai_scores.extend(batch_scores)
                openai_raw.extend(batch_raw)
                
                # Rate limiting pause
                if i + batch_size < len(df):
                    import time
                    time.sleep(1)
            
            df["openai_score"] = openai_scores
            df["openai_raw"] = openai_raw
            
            print(f"✓ OpenAI analysis complete")
            print(f"  Average score: {df['openai_score'].mean():.3f}")
            print(f"  Sentiment range: [{df['openai_score'].min():.3f}, {df['openai_score'].max():.3f}]")
            
            # Distribution
            positive = (df['openai_score'] > 0.1).sum()
            negative = (df['openai_score'] < -0.1).sum()
            neutral = len(df) - positive - negative
            
            print(f"  Positive: {positive} ({positive/len(df)*100:.1f}%)")
            print(f"  Negative: {negative} ({negative/len(df)*100:.1f}%)")
            print(f"  Neutral: {neutral} ({neutral/len(df)*100:.1f}%)")
            
        else:
            print("⚠️ Skipping OpenAI analysis (no API key)")
            # Add placeholder columns
            df["openai_score"] = 0.0
            df["openai_raw"] = "Skipped - no API key"
            
    except ImportError:
        print("⚠️ OpenAI library not installed")
        df["openai_score"] = 0.0
        df["openai_raw"] = "Skipped - library not available"

## 5) Configurable Smoothing Analysis

**Advanced smoothing** with multiple methods and adjustable parameters.

In [ ]:
if 'df' not in globals():
    print("⚠️ Please process text and run sentiment analysis first")
else:
    print("🔧 Setting up configurable smoothing analysis...")
    
    # Enhanced smoothing functions
    def z_score(x: np.ndarray) -> np.ndarray:
        """Robust standardization"""
        x = np.asarray(x, dtype=float)
        mean_val = np.nanmean(x)
        std_val = np.nanstd(x)
        if std_val < 1e-8:
            return np.zeros_like(x)
        return (x - mean_val) / std_val
    
    def moving_average(x, k):
        """Moving average with edge handling"""
        k = max(1, int(k))
        if k >= len(x):
            return np.full_like(x, np.mean(x))
        return np.convolve(x, np.ones(k) / k, mode="same")
    
    def exponential_moving_average(x, alpha):
        """EMA with validation"""
        if not 0 < alpha <= 1:
            raise ValueError(f"Alpha must be in (0,1], got {alpha}")
        
        x = np.asarray(x, dtype=float)
        y = np.zeros_like(x)
        y[0] = x[0]
        for t in range(1, len(x)):
            y[t] = alpha * x[t] + (1 - alpha) * y[t-1]
        return y
    
    def savgol_smooth(x, window_length, polyorder):
        """Savitzky-Golay with robust error handling"""
        try:
            from scipy.signal import savgol_filter
            x = np.asarray(x, dtype=float)
            
            if len(x) < 3:
                return moving_average(x, min(len(x), 3))
            
            wl = max(3, int(window_length) | 1)
            wl = min(wl, len(x))
            poly = min(polyorder, wl-1)
            poly = max(1, poly)
            
            return savgol_filter(x, wl, poly, mode="interp")
        except Exception:
            return moving_average(x, max(3, int(window_length)))
    
    # Prepare sentiment columns (standardize)
    sentiment_cols = []
    if 'vader' in df.columns:
        df["vader_z"] = z_score(df["vader"].values)
        sentiment_cols.append("vader_z")
    
    if 'bert_score' in df.columns:
        df["bert_z"] = z_score(df["bert_score"].values)
        sentiment_cols.append("bert_z")
    
    if 'openai_score' in df.columns:
        df["openai_z"] = z_score(df["openai_score"].values)
        sentiment_cols.append("openai_z")
    
    # Calculate ensemble sentiment
    if len(sentiment_cols) > 0:
        sentiment_matrix = df[sentiment_cols].values.T  # shape: (n_methods, n_sentences)
        df["ensemble_z"] = z_score(np.mean(sentiment_matrix, axis=0))
        print(f"✓ Ensemble sentiment calculated from {len(sentiment_cols)} methods")
    else:
        print("⚠️ No sentiment columns found")
        
    print(f"✓ Sentiment columns standardized: {sentiment_cols}")
    print(f"✓ Ready for configurable smoothing analysis")

In [ ]:
if 'df' not in globals() or 'ensemble_z' not in df.columns:
    print("⚠️ Please complete sentiment analysis first")
else:
    print("🎛️ Running configurable smoothing analysis...")
    
    # Generate all smoothing combinations
    smoothing_results = {}
    
    base_series = df["ensemble_z"].values
    n_sentences = len(base_series)
    
    print(f"Analyzing {n_sentences} sentences with {len(CFG.smoothing_methods)} methods...")
    
    # Moving Average variations
    if "ma" in CFG.smoothing_methods:
        for window_frac in CFG.window_sizes:
            window_size = max(3, int(np.ceil(n_sentences * window_frac)) | 1)
            window_size = min(window_size, n_sentences)
            
            smooth_ma = moving_average(base_series, window_size)
            key = f"ma_w{window_frac}"
            smoothing_results[key] = smooth_ma
            df[key] = smooth_ma
            print(f"  ✓ MA: window={window_frac} (size={window_size})")
    
    # Exponential Moving Average variations
    if "ema" in CFG.smoothing_methods:
        for alpha in CFG.ema_alphas:
            smooth_ema = exponential_moving_average(base_series, alpha)
            key = f"ema_a{alpha}"
            smoothing_results[key] = smooth_ema
            df[key] = smooth_ema
            print(f"  ✓ EMA: alpha={alpha}")
    
    # Savitzky-Golay variations
    if "savgol" in CFG.smoothing_methods:
        for window_frac in CFG.window_sizes:
            for poly in CFG.savgol_polys:
                window_size = max(3, int(np.ceil(n_sentences * window_frac)) | 1)
                window_size = min(window_size, n_sentences)
                
                smooth_savgol = savgol_smooth(base_series, window_size, poly)
                key = f"savgol_w{window_frac}_p{poly}"
                smoothing_results[key] = smooth_savgol
                df[key] = smooth_savgol
                print(f"  ✓ SavGol: window={window_frac}, poly={poly}")
    
    print(f"\n✓ Generated {len(smoothing_results)} smoothing variations")
    
    # Store results for later analysis
    smoothing_config = {
        'results': smoothing_results,
        'base_series': base_series,
        'methods': list(smoothing_results.keys())
    }
    
    print("✓ Configurable smoothing analysis complete")

## 6) Interactive Smoothing Comparison

**Interactive visualization** to compare different smoothing parameters.

In [ ]:
if 'df' not in globals() or 'smoothing_config' not in globals():
    print("⚠️ Please complete smoothing analysis first")
else:
    import matplotlib.pyplot as plt
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.offline as pyo
    
    print("📊 Creating interactive smoothing comparison...")
    
    # Create comparison widget
    method_selector = widgets.SelectMultiple(
        options=smoothing_config['methods'],
        value=smoothing_config['methods'][:4],  # Select first 4 by default
        description='Smoothing Methods:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px', height='200px')
    )
    
    compare_button = widgets.Button(
        description='Compare Selected',
        button_style='primary'
    )
    
    plot_output = widgets.Output()
    
    def compare_smoothing(change=None):
        with plot_output:
            clear_output()
            
            selected_methods = list(method_selector.value)
            if not selected_methods:
                print("Please select at least one smoothing method")
                return
            
            print(f"Comparing {len(selected_methods)} smoothing methods...")
            
            # Create static plot
            fig, ax = plt.subplots(figsize=(14, 6))
            
            # Plot original data
            x = df['i'].values
            ax.plot(x, smoothing_config['base_series'], 
                   label='Original', alpha=0.3, linewidth=1, color='gray')
            
            # Plot selected smoothing methods
            colors = plt.cm.tab10(np.linspace(0, 1, len(selected_methods)))
            
            for i, method in enumerate(selected_methods):
                if method in smoothing_config['results']:
                    ax.plot(x, smoothing_config['results'][method],
                           label=method, linewidth=2, color=colors[i], alpha=0.8)
            
            ax.set_xlabel('Sentence Number')
            ax.set_ylabel('Sentiment (z-score)')
            ax.set_title('Sentiment Smoothing Comparison')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Create interactive plot if enabled
            if CFG.interactive_plots:
                fig_interactive = go.Figure()
                
                # Add original data
                fig_interactive.add_trace(go.Scatter(
                    x=x, y=smoothing_config['base_series'],
                    mode='lines', name='Original',
                    line=dict(width=1, color='gray'),
                    opacity=0.5
                ))
                
                # Add selected methods
                for i, method in enumerate(selected_methods):
                    if method in smoothing_config['results']:
                        fig_interactive.add_trace(go.Scatter(
                            x=x, y=smoothing_config['results'][method],
                            mode='lines', name=method,
                            line=dict(width=2),
                            hovertext=df['sentence'],
                            hoverinfo='text+x+y'
                        ))
                
                fig_interactive.update_layout(
                    title='Interactive Sentiment Smoothing Comparison',
                    xaxis_title='Sentence Number',
                    yaxis_title='Sentiment (z-score)',
                    height=500
                )
                
                fig_interactive.show()
            
            # Calculate and display statistics
            print("\n📈 Smoothing Statistics:")
            print("=" * 40)
            
            for method in selected_methods:
                if method in smoothing_config['results']:
                    smooth_data = smoothing_config['results'][method]
                    
                    # Calculate smoothness (lower variance = smoother)
                    smoothness = np.var(np.diff(smooth_data))
                    
                    # Calculate preservation of original pattern
                    correlation = np.corrcoef(smoothing_config['base_series'], smooth_data)[0,1]
                    
                    print(f"{method}:")
                    print(f"  Smoothness: {smoothness:.4f} (lower = smoother)")
                    print(f"  Correlation with original: {correlation:.3f}")
                    print(f"  Range: [{smooth_data.min():.3f}, {smooth_data.max():.3f}]")
                    print()
    
    compare_button.on_click(compare_smoothing)
    
    # Display interface
    print("🎛️ Select smoothing methods to compare:")
    display(method_selector)
    display(compare_button)
    display(plot_output)

## 7) Peak/Valley Detection with Context Extraction

**Advanced detection** with configurable parameters and CRUX_WIDTH context extraction.

In [ ]:
if 'df' not in globals() or 'smoothing_config' not in globals():
    print("⚠️ Please complete smoothing analysis first")
else:
    print("🔍 Running peak/valley detection with context extraction...")
    
    from math import ceil
    import warnings
    
    # Use best smoothing method (default to first Savitzky-Golay if available)
    default_method = None
    for method in smoothing_config['methods']:
        if method.startswith('savgol'):
            default_method = method
            break
    
    if not default_method:
        default_method = smoothing_config['methods'][0]
    
    print(f"Using smoothing method: {default_method}")
    
    # Get the smoothed series
    smoothed_series = smoothing_config['results'][default_method]
    
    # Peak detection
    try:
        from scipy.signal import find_peaks
        
        # Detect peaks
        peak_distance = max(1, int(ceil(len(df) * CFG.peak_min_frac)))
        peaks, peak_properties = find_peaks(
            smoothed_series, 
            distance=peak_distance,
            prominence=CFG.peak_prominence
        )
        
        # Detect valleys (peaks on inverted signal)
        valleys, valley_properties = find_peaks(
            -smoothed_series,
            distance=peak_distance,
            prominence=CFG.peak_prominence
        )
        
        print(f"✓ Using scipy.signal.find_peaks")
        
    except Exception:
        # Fallback peak detection
        def simple_peaks(x, distance):
            x = np.asarray(x)
            peaks = []
            for i in range(1, len(x)-1):
                if x[i] > x[i-1] and x[i] > x[i+1]:
                    peaks.append(i)
            
            # Filter by distance
            filtered = []
            for p in peaks:
                if not filtered or p - filtered[-1] >= distance:
                    filtered.append(p)
            return np.array(filtered)
        
        peak_distance = max(1, int(ceil(len(df) * CFG.peak_min_frac)))
        peaks = simple_peaks(smoothed_series, peak_distance)
        valleys = simple_peaks(-smoothed_series, peak_distance)
        print(f"✓ Using fallback peak detection")
    
    def extract_context(idx, width=CFG.CRUX_WIDTH):
        """Extract context around a peak/valley"""
        start_idx = max(0, idx - width)
        end_idx = min(len(df), idx + width + 1)
        
        context_sentences = df.iloc[start_idx:end_idx]['sentence'].tolist()
        context_indices = list(range(start_idx, end_idx))
        
        return {
            'peak_index': idx,
            'peak_sentence': df.iloc[idx]['sentence'],
            'context_sentences': context_sentences,
            'context_indices': context_indices,
            'context_start': start_idx,
            'context_end': end_idx - 1,
            'context_width': len(context_sentences)
        }
    
    # Extract peak contexts
    peak_contexts = []
    for peak_idx in peaks:
        context = extract_context(peak_idx)
        context['sentiment_value'] = smoothed_series[peak_idx]
        context['type'] = 'peak'
        peak_contexts.append(context)
    
    # Extract valley contexts
    valley_contexts = []
    for valley_idx in valleys:
        context = extract_context(valley_idx)
        context['sentiment_value'] = smoothed_series[valley_idx]
        context['type'] = 'valley'
        valley_contexts.append(context)
    
    # Store results
    peak_valley_data = {
        'peaks': peaks,
        'valleys': valleys,
        'peak_contexts': peak_contexts,
        'valley_contexts': valley_contexts,
        'smoothing_method': default_method,
        'crux_width': CFG.CRUX_WIDTH
    }
    
    print(f"✓ Detected {len(peaks)} peaks and {len(valleys)} valleys")
    print(f"✓ Extracted context with width ±{CFG.CRUX_WIDTH} sentences")
    
    # Display summary
    if peaks.size > 0:
        peak_values = smoothed_series[peaks]
        print(f"  Peak sentiment range: [{peak_values.min():.3f}, {peak_values.max():.3f}]")
    
    if valleys.size > 0:
        valley_values = smoothed_series[valleys]
        print(f"  Valley sentiment range: [{valley_values.min():.3f}, {valley_values.max():.3f}]")
    
    print(f"✓ Peak/valley detection complete")

### 7.1 Peak/Valley Visualization with Context

In [ ]:
if 'peak_valley_data' not in globals():
    print("⚠️ Please complete peak/valley detection first")
else:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    
    print("📊 Creating peak/valley visualization with context...")
    
    # Create comprehensive plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), 
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    x = df['i'].values
    
    # Main sentiment plot
    ax1.plot(x, smoothing_config['base_series'], 
            label='Original Sentiment', alpha=0.3, linewidth=1, color='gray')
    ax1.plot(x, smoothing_config['results'][peak_valley_data['smoothing_method']],
            label=f'Smoothed ({peak_valley_data["smoothing_method"]})', 
            linewidth=2.5, color='blue')
    
    # Mark peaks
    if len(peak_valley_data['peaks']) > 0:
        peak_x = peak_valley_data['peaks']
        peak_y = smoothing_config['results'][peak_valley_data['smoothing_method']][peak_x]
        ax1.scatter(peak_x, peak_y, marker='^', s=100, color='red', 
                   label=f'Peaks ({len(peak_x)})', zorder=10, linewidth=2)
        
        # Add peak labels
        for i, (px, py) in enumerate(zip(peak_x, peak_y)):
            ax1.annotate(f'P{i+1}', (px, py), xytext=(5, 5), 
                       textcoords='offset points', fontsize=8, color='red')
    
    # Mark valleys
    if len(peak_valley_data['valleys']) > 0:
        valley_x = peak_valley_data['valleys']
        valley_y = smoothing_config['results'][peak_valley_data['smoothing_method']][valley_x]
        ax1.scatter(valley_x, valley_y, marker='v', s=100, color='green',
                   label=f'Valleys ({len(valley_x)})', zorder=10, linewidth=2)
        
        # Add valley labels
        for i, (vx, vy) in enumerate(zip(valley_x, valley_y)):
            ax1.annotate(f'V{i+1}', (vx, vy), xytext=(5, -5), 
                       textcoords='offset points', fontsize=8, color='green')
    
    # Highlight context regions
    for context in peak_valley_data['peak_contexts']:
        start, end = context['context_start'], context['context_end']
        ax1.axvspan(start, end, alpha=0.1, color='red')
    
    for context in peak_valley_data['valley_contexts']:
        start, end = context['context_start'], context['context_end']
        ax1.axvspan(start, end, alpha=0.1, color='green')
    
    ax1.set_xlabel('Sentence Number')
    ax1.set_ylabel('Sentiment (z-score)')
    ax1.set_title(f'Sentiment Arc with Peak/Valley Detection\nContext Width: ±{CFG.CRUX_WIDTH} sentences')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    
    # Context detail plot
    ax2.set_title('Peak/Valley Context Regions')
    ax2.set_xlabel('Sentence Number')
    ax2.set_ylabel('')
    
    # Show context regions on detail plot
    y_pos = 0
    
    for i, context in enumerate(peak_valley_data['peak_contexts']):
        start, end = context['context_start'], context['context_end']
        ax2.barh(y_pos, end-start+1, left=start, height=0.3, 
                color='red', alpha=0.6, label='Peak' if i == 0 else '')
        ax2.text(start + (end-start)/2, y_pos, f'P{i+1}', 
                ha='center', va='center', fontsize=8, color='white', weight='bold')
    
    y_pos += 0.5
    
    for i, context in enumerate(peak_valley_data['valley_contexts']):
        start, end = context['context_start'], context['context_end']
        ax2.barh(y_pos, end-start+1, left=start, height=0.3, 
                color='green', alpha=0.6, label='Valley' if i == 0 else '')
        ax2.text(start + (end-start)/2, y_pos, f'V{i+1}', 
                ha='center', va='center', fontsize=8, color='white', weight='bold')
    
    ax2.set_ylim(-0.5, y_pos + 0.5)
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Visualization complete with {len(peak_valley_data['peak_contexts'])} peak contexts and {len(peak_valley_data['valley_contexts'])} valley contexts")

## 8) Downloadable Report Generation

**Comprehensive reports** with peak/valley analysis and context extraction.

In [ ]:
if 'peak_valley_data' not in globals():
    print("⚠️ Please complete peak/valley detection first")
else:
    print("📄 Generating downloadable reports...")
    
    import json
    from datetime import datetime
    
    # Create timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Generate comprehensive report
    report_data = {
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'total_sentences': len(df),
            'smoothing_method': peak_valley_data['smoothing_method'],
            'crux_width': CFG.CRUX_WIDTH,
            'peak_detection_params': {
                'min_fraction': CFG.peak_min_frac,
                'prominence': CFG.peak_prominence
            }
        },
        'summary': {
            'peaks_detected': len(peak_valley_data['peaks']),
            'valleys_detected': len(peak_valley_data['valleys']),
            'peak_sentiment_range': None,
            'valley_sentiment_range': None
        },
        'peaks': [],
        'valleys': []
    }
    
    # Add peak ranges
    if len(peak_valley_data['peaks']) > 0:
        peak_values = smoothing_config['results'][peak_valley_data['smoothing_method']][peak_valley_data['peaks']]
        report_data['summary']['peak_sentiment_range'] = [float(peak_values.min()), float(peak_values.max())]
    
    if len(peak_valley_data['valleys']) > 0:
        valley_values = smoothing_config['results'][peak_valley_data['smoothing_method']][peak_valley_data['valleys']]
        report_data['summary']['valley_sentiment_range'] = [float(valley_values.min()), float(valley_values.max())]
    
    # Process peak contexts
    for i, context in enumerate(peak_valley_data['peak_contexts']):
        peak_data = {
            'peak_id': f'P{i+1}',
            'sentence_index': int(context['peak_index']),
            'sentiment_value': float(context['sentiment_value']),
            'peak_sentence': context['peak_sentence'],
            'context': {
                'start_index': int(context['context_start']),
                'end_index': int(context['context_end']),
                'total_sentences': int(context['context_width']),
                'sentences': context['context_sentences']
            }
        }
        report_data['peaks'].append(peak_data)
    
    # Process valley contexts
    for i, context in enumerate(peak_valley_data['valley_contexts']):
        valley_data = {
            'valley_id': f'V{i+1}',
            'sentence_index': int(context['peak_index']),
            'sentiment_value': float(context['sentiment_value']),
            'valley_sentence': context['peak_sentence'],
            'context': {
                'start_index': int(context['context_start']),
                'end_index': int(context['context_end']),
                'total_sentences': int(context['context_width']),
                'sentences': context['context_sentences']
            }
        }
        report_data['valleys'].append(valley_data)
    
    # Save JSON report
    json_filename = out_dir / f"sentiment_report_{timestamp}.json"
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(report_data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ JSON report saved: {json_filename}")
    
    # Generate text report
    text_filename = out_dir / f"sentiment_analysis_{timestamp}.txt"
    
    with open(text_filename, 'w', encoding='utf-8') as f:
        f.write("SENTIMENT ARC ANALYSIS REPORT\n")
        f.write("=" * 50 + "\n\n")
        
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Sentences: {len(df)}\n")
        f.write(f"Smoothing Method: {peak_valley_data['smoothing_method']}\n")
        f.write(f"Context Width: ±{CFG.CRUX_WIDTH} sentences\n\n")
        
        f.write("SUMMARY\n")
        f.write("-" * 20 + "\n")
        f.write(f"Peaks Detected: {len(peak_valley_data['peaks'])}\n")
        f.write(f"Valleys Detected: {len(peak_valley_data['valleys'])}\n\n")
        
        # Peak details
        if peak_valley_data['peak_contexts']:
            f.write("PEAKS (High Sentiment Points)\n")
            f.write("-" * 35 + "\n\n")
            
            for i, context in enumerate(peak_valley_data['peak_contexts']):
                f.write(f"PEAK {i+1} (Sentence {context['peak_index'] + 1})\n")
                f.write(f"Sentiment Value: {context['sentiment_value']:.3f}\n")
                f.write(f"Peak Sentence: {context['peak_sentence']}\n")
                f.write(f"Context (Sentences {context['context_start'] + 1}-{context['context_end'] + 1}):\n")
                
                for j, sent in enumerate(context['context_sentences']):
                    sent_idx = context['context_start'] + j
                    marker = " >>>>>" if sent_idx == context['peak_index'] else ""
                    f.write(f"  {sent_idx + 1:3d}: {sent}{marker}\n")
                
                f.write("\n" + "-" * 50 + "\n\n")
        
        # Valley details
        if peak_valley_data['valley_contexts']:
            f.write("VALLEYS (Low Sentiment Points)\n")
            f.write("-" * 37 + "\n\n")
            
            for i, context in enumerate(peak_valley_data['valley_contexts']):
                f.write(f"VALLEY {i+1} (Sentence {context['peak_index'] + 1})\n")
                f.write(f"Sentiment Value: {context['sentiment_value']:.3f}\n")
                f.write(f"Valley Sentence: {context['peak_sentence']}\n")
                f.write(f"Context (Sentences {context['context_start'] + 1}-{context['context_end'] + 1}):\n")
                
                for j, sent in enumerate(context['context_sentences']):
                    sent_idx = context['context_start'] + j
                    marker = " >>>>>" if sent_idx == context['peak_index'] else ""
                    f.write(f"  {sent_idx + 1:3d}: {sent}{marker}\n")
                
                f.write("\n" + "-" * 50 + "\n\n")
    
    print(f"✓ Text report saved: {text_filename}")
    
    # Generate CSV with all sentiment data
    csv_filename = out_dir / f"sentiment_data_{timestamp}.csv"
    
    # Prepare columns for CSV
    csv_columns = ['i', 'sentence', 'sentence_length']
    
    # Add sentiment columns
    for col in ['vader', 'vader_pos', 'vader_neg', 'vader_neu']:
        if col in df.columns:
            csv_columns.append(col)
    
    for col in ['bert_score', 'bert_label', 'bert_confidence']:
        if col in df.columns:
            csv_columns.append(col)
    
    if 'openai_score' in df.columns:
        csv_columns.append('openai_score')
    
    # Add standardized columns
    for col in ['vader_z', 'bert_z', 'openai_z', 'ensemble_z']:
        if col in df.columns:
            csv_columns.append(col)
    
    # Add smoothing columns
    smoothing_cols = [col for col in df.columns if any(pattern in col for pattern in ['ma_w', 'ema_a', 'savgol_'])]
    csv_columns.extend(smoothing_cols[:10])  # Limit to first 10 to avoid overly wide CSV
    
    # Mark peaks and valleys
    df['is_peak'] = df.index.isin(peak_valley_data['peaks'])
    df['is_valley'] = df.index.isin(peak_valley_data['valleys'])
    csv_columns.extend(['is_peak', 'is_valley'])
    
    # Save CSV
    df[csv_columns].to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"✓ CSV data saved: {csv_filename}")
    
    # Create download links
    print(f"\n📁 All reports saved to: {out_dir}")
    print(f"\n📋 Report Files Generated:")
    print(f"  • JSON Report: {json_filename.name}")
    print(f"  • Text Report: {text_filename.name}")
    print(f"  • CSV Data: {csv_filename.name}")
    
    # Display summary
    print(f"\n📊 Analysis Summary:")
    print(f"  • Total Sentences: {len(df):,}")
    print(f"  • Peaks Found: {len(peak_valley_data['peaks'])}")
    print(f"  • Valleys Found: {len(peak_valley_data['valleys'])}")
    print(f"  • Context Width: ±{CFG.CRUX_WIDTH} sentences")
    print(f"  • Smoothing Method: {peak_valley_data['smoothing_method']}")
    
    if peak_valley_data['peak_contexts']:
        print(f"  • Peak Contexts: {len(peak_valley_data['peak_contexts'])} extracted")
    if peak_valley_data['valley_contexts']:
        print(f"  • Valley Contexts: {len(peak_valley_data['valley_contexts'])} extracted")

## 9) Summary and Export Options

**Final summary** with export options and analysis completion.

In [ ]:
print("🎉 SENTIMENT ARC ANALYSIS COMPLETE")
print("=" * 60)

# Final summary statistics
if 'df' in globals():
    print(f"\n📊 DATASET SUMMARY:")
    print(f"  • Total sentences processed: {len(df):,}")
    print(f"  • Average sentence length: {df['sentence_length'].mean():.1f} characters")
    print(f"  • Total text length: {df['sentence_length'].sum():,} characters")
    
    # Sentiment method summary
    methods_available = []
    if 'vader' in df.columns:
        methods_available.append("VADER")
    if 'bert_score' in df.columns:
        methods_available.append("DistilBERT")
    if 'openai_score' in df.columns:
        methods_available.append("OpenAI API")
    
    print(f"  • Sentiment methods used: {', '.join(methods_available)}")
    
    if 'ensemble_z' in df.columns:
        print(f"  • Ensemble sentiment range: [{df['ensemble_z'].min():.3f}, {df['ensemble_z'].max():.3f}]")
    
# Smoothing analysis summary
if 'smoothing_config' in globals():
    print(f"\n🔧 SMOOTHING ANALYSIS:")
    print(f"  • Smoothing variations generated: {len(smoothing_config['methods'])}")
    print(f"  • Methods used: {', ', join(CFG.smoothing_methods)}")
    
# Peak/valley summary
if 'peak_valley_data' in globals():
    print(f"\n🔍 PEAK/VALLEY ANALYSIS:")
    print(f"  • Peaks detected: {len(peak_valley_data['peaks'])}")
    print(f"  • Valleys detected: {len(peak_valley_data['valleys'])}")
    print(f"  • Context extraction width: ±{CFG.CRUX_WIDTH} sentences")
    print(f"  • Total contexts extracted: {len(peak_valley_data['peak_contexts']) + len(peak_valley_data['valley_contexts'])}")
    
    # Show most significant peaks and valleys
    if peak_valley_data['peak_contexts']:
        # Sort peaks by sentiment value
        sorted_peaks = sorted(peak_valley_data['peak_contexts'], 
                             key=lambda x: x['sentiment_value'], reverse=True)
        print(f"\n📈 TOP 3 PEAKS:")
        for i, peak in enumerate(sorted_peaks[:3]):
            print(f"  {i+1}. Sentence {peak['peak_index']+1}: {peak['sentiment_value']:.3f}")
            print(f"     "{peak['peak_sentence'][:80]}..."")
    
    if peak_valley_data['valley_contexts']:
        # Sort valleys by sentiment value (most negative first)
        sorted_valleys = sorted(peak_valley_data['valley_contexts'], 
                               key=lambda x: x['sentiment_value'])
        print(f"\n📉 TOP 3 VALLEYS:")
        for i, valley in enumerate(sorted_valleys[:3]):
            print(f"  {i+1}. Sentence {valley['peak_index']+1}: {valley['sentiment_value']:.3f}")
            print(f"     "{valley['peak_sentence'][:80]}..."")

# Export summary
print(f"\n📁 EXPORT SUMMARY:")
print(f"  • Output directory: {out_dir}")

# List generated files
if out_dir.exists():
    generated_files = list(out_dir.glob("*"))
    if generated_files:
        print(f"  • Files generated: {len(generated_files)}")
        for file in sorted(generated_files):
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"    - {file.name} ({size_mb:.2f} MB)")
    else:
        print(f"  • No files generated yet")

# Configuration summary
print(f"\n⚙️ CONFIGURATION USED:")
print(f"  • CRUX_WIDTH: {CFG.CRUX_WIDTH}")
print(f"  • Peak minimum fraction: {CFG.peak_min_frac}")
print(f"  • Peak prominence threshold: {CFG.peak_prominence}")
print(f"  • Maximum sentences: {CFG.max_sentences}")
print(f"  • Random seed: {CFG.seed}")

print(f"\n✨ Analysis complete! Check the output directory for detailed reports.")

# Provide quick access to key variables
print(f"\n🔑 KEY VARIABLES FOR FURTHER ANALYSIS:")
key_vars = {
    'df': 'Main DataFrame with all sentiment data',
    'smoothing_config': 'All smoothing variations',
    'peak_valley_data': 'Peak/valley detection results with contexts'
}

for var_name, description in key_vars.items():
    if var_name in globals():
        print(f"  • {var_name}: {description}")
    else:
        print(f"  • {var_name}: Not available")

---

## Usage Instructions:

1. **Upload Text**: Use the file upload widget or paste text directly
2. **Run Analysis**: Execute all sentiment analysis cells (VADER, DistilBERT, OpenAI)
3. **Configure Smoothing**: Adjust parameters in the EnhancedConfig cell if needed
4. **Compare Methods**: Use the interactive comparison widget
5. **Extract Peaks/Valleys**: Automatic detection with context extraction
6. **Download Reports**: JSON, text, and CSV files are generated automatically

### Key Features:
- **CRUX_WIDTH**: Adjust context extraction width (default: 5 sentences)
- **Multiple APIs**: VADER (fast), DistilBERT (balanced), OpenAI (high quality)
- **Configurable Smoothing**: Test multiple parameters for optimal results
- **Context Preservation**: Peak/valley contexts maintain narrative flow
- **Multiple Export Formats**: JSON for programmatic use, text for reading, CSV for analysis

### Notes:
- OpenAI API requires valid key (set OPENAI_API_KEY environment variable)
- Large texts are automatically truncated for API efficiency
- All visualizations include both static and interactive versions
- Reports include full sentence context for each detected peak/valley